# 07 · Robustness and sensitivity

**Purpose.** Check whether the forecast comparison of notebook 06 depends on three choices made along the way: scoring a single lowest half hour, the reporting threshold used to build the network total, and the level steps between Ausgrid's reporting-year files.

**Paper section served.** IV-B Results: forecast comparison (robustness).

**Inputs**

- `data/interim/forecasts.parquet` and `actuals.parquet` from notebook 05.
- `data/interim/network_15min.parquet` and `network_30min.parquet` from notebook 01.
- `data/interim/pv_daily.parquet` and `weather_daily.parquet` from notebook 02.

**Paper artifacts.** None.

**Supporting checks** (`results/checks`)

- `table3_robust.csv`: Table III on the robustness target (the mean of the 15 lowest daily minima).
- `sensitivity_robust.csv`: the comparisons on the whole-day minimum and on the robustness target.
- `sensitivity_thresholds.csv`: the annual lows at each reporting threshold.
- `sensitivity_rescale_factors.csv` and `sensitivity_rescaled_lows.csv`: the factor applied to each reporting-year file, and the annual lows under the published and the rescaled series.
- `table3_daytime_low_rescaled.csv` and `table3_minimum_rescaled.csv`: Table III on the rescaled series.
- `sensitivity_rescaled_comparisons.csv`: the comparisons under both series.

**Data for later use** (`data/interim`)

- `forecasts_rescaled.parquet`: the backtest of all six models on the rescaled series.

**Main process**

1. Set up.
2. One extreme half hour: Table III on the robustness target.
3. The reporting threshold: annual lows at 90%, 95% and 98%.
4. The level steps between reporting years: a rescaled series.
5. Result.

**The comparisons.** The same four as in notebook 06. H1 (M3 against M2) and H2 (M4 against M3) were fixed before any model was run and are judged on the daytime low. Model form is M5 against M4. The supplementary comparison, M5 against M6, was added after the scores of M1 to M5 had been seen. A negative change means the first model of the pair scores better.

## 1. Set up

In [1]:
import pandas as pd

from pynmdf import clean, evaluate, targets
from pynmdf.config import load_config

cfg = load_config()
interim, checks = cfg["dirs"]["interim"], cfg["dirs"]["checks"]
study, wanted = cfg["study"], cfg["targets"]

pd.set_option("display.width", 220, "display.max_columns", 40)
pd.set_option("display.float_format", "{:,.1f}".format)
forecasts = pd.read_parquet(interim / "forecasts.parquet")
actuals = pd.read_parquet(interim / "actuals.parquet")
network = pd.read_parquet(interim / "network_15min.parquet")
total = pd.read_parquet(interim / "network_30min.parquet")["total_mw"]
pv = pd.read_parquet(interim / "pv_daily.parquet")["pv_mw"]
weather = pd.read_parquet(interim / "weather_daily.parquet")

# The two targets reported in the paper, and the columns of Table III shown on screen.
REPORTED = ["daytime_low", "minimum"]
LEVELS = ["annual", "monthly"]
SHOWN = [f"{level}_{name}" for level in LEVELS for name in ("pinball_mean", "coverage")]

## 2. One extreme half hour: Table III on the robustness target

The whole-day minimum is a single half hour, so one unusual reading could decide a year's score. The robustness target replaces it with the mean of the 15 lowest daily minima of the year (the 2 lowest in a month), taken over complete days only. Every model forecasts it directly, from the same simulations as the minimum. If the models rank the same way on both, the ranking does not rest on one half hour.

The first table is Table III on the robustness target. `points` is the number of test points scored.

In [2]:
table3_robust = evaluate.score_table(forecasts, "robust")
table3_robust.to_csv(checks /"table3_robust.csv")
table3_robust[[*SHOWN, "annual_points", "monthly_points"]]

,annual_pinball_mean,annual_coverage,monthly_pinball_mean,monthly_coverage,annual_points,monthly_points
model,,,,,,
M1,36.9,0.3,40.5,0.4,7,76
M2,25.8,1.0,34.4,0.7,7,76
M3,31.9,0.7,39.3,0.6,7,76
M4,35.9,0.6,41.1,0.6,7,76
M5,25.5,0.6,30.2,0.6,7,76
M6,30.5,0.7,39.6,0.7,7,76


The next table puts the mean pinball loss (MW) on the whole-day minimum beside that on the robustness target, and the one after it does the same for the four comparisons (change in mean pinball loss, in per cent).

In [3]:
columns = [(level, target) for level in LEVELS for target in ("minimum", "robust")]
losses = evaluate.score(forecasts, ["level", "target", "model"])["pinball_mean"]
losses.unstack(["level", "target"])[columns]

level   annual        monthly       
target minimum robust minimum robust
model                               
M1        45.2   36.9    43.5   40.5
M2        31.1   25.8    37.5   34.4
M3        41.6   31.9    41.4   39.3
M4        60.0   35.9    44.1   41.1
M5        26.9   25.5    32.5   30.2
M6        28.0   30.5    42.2   39.6

In [4]:
on_both = forecasts[forecasts["target"].isin(["minimum", "robust"])]
robust_changes = evaluate.compare(on_both, evaluate.COMPARISONS)["change_pct"]
robust_changes = robust_changes.unstack(["level", "target"])[columns]
robust_changes.to_csv(checks /"sensitivity_robust.csv")
robust_changes

level                                annual        monthly       
target                              minimum robust minimum robust
comparison                                                       
H1: PV against time                    34.1   23.3    10.3   14.3
H2: weather on top of PV               44.2   12.6     6.6    4.7
Model form: XGBoost against linear    -55.2  -29.0   -26.3  -26.5
Supplementary: weather in tree form    -4.0  -16.6   -23.0  -23.7

The robustness target is built from complete days only, so a year with many incomplete days draws its 15 lowest from fewer days. The table gives, for each year, the whole-day minimum, the robustness value and the number of complete days behind it.

In [5]:
annual = actuals[actuals["level"] == "annual"]
days = annual[annual["target"] == "robust"].set_index("year")["complete_days"]
beside = annual.pivot(index="year", columns="target", values="actual")[["minimum", "robust"]]
beside.assign(complete_days=days).T.round()

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
target,,,,,,,,,,,
minimum,"1,368.0","1,433.0","1,474.0","1,495.0","1,473.0","1,429.0","1,357.0","1,428.0","1,080.0","1,019.0",880.0
robust,"1,403.0","1,448.0","1,508.0","1,538.0","1,525.0","1,461.0","1,408.0","1,467.0","1,218.0","1,120.0","1,024.0"
complete_days,357.0,361.0,360.0,365.0,353.0,347.0,331.0,352.0,342.0,309.0,359.0


## 3. The reporting threshold: annual lows at 90%, 95% and 98%

An interval of the network total is kept only when the substations reporting in it carry at least 95% of usual load. A looser threshold keeps more intervals, each resting on a larger scale-up; a stricter one keeps fewer. If a year's low came from a poorly reported interval, it would move with the threshold.

The network total is rebuilt at 90%, 95% and 98% from the cached 15-minute series. The table gives each year's whole-day minimum and daytime low (MW) at the three thresholds, and the share of half hours kept (per cent). The 95% column is the series used everywhere else.

In [6]:
def annual_lows(series: pd.Series) -> pd.DataFrame:
    """Each year's whole-day minimum and daytime low (MW) and the half hours kept (%)."""
    values = targets.period_targets(
        series, wanted["troughs"], wanted["month_min_kept_share"], wanted["robust_days"]
    )
    yearly = values[values["level"] == "annual"]
    lows = yearly.pivot(index="year", columns="target", values="actual")[REPORTED]
    kept = yearly[yearly["target"] == "minimum"].set_index("year")["kept_share"]
    return lows.assign(kept_pct=100 * kept)


by_threshold = {}
for share in cfg["cleaning"]["sensitivity_reporting_shares"]:
    rebuilt = clean.half_hourly_total(network, share, study["first_year"], study["last_year"])
    by_threshold[f"{share:.0%}"] = annual_lows(rebuilt)
by_threshold = pd.concat(by_threshold, axis=1).swaplevel(axis=1)
by_threshold = by_threshold[["minimum", "daytime_low", "kept_pct"]]
by_threshold.to_csv(checks /"sensitivity_thresholds.csv")

# The 95% rebuild must be the series the rest of the analysis used.
at_95 = by_threshold.xs("95%", axis=1, level=1)
pd.testing.assert_frame_equal(at_95, annual_lows(total), check_like=True)
by_threshold.round(1)

target minimum                 daytime_low                 kept_pct           
           90%     95%     98%         90%     95%     98%      90%   95%  98%
year                                                                          
2015   1,368.1 1,368.1 1,368.1     1,832.6 1,832.6 1,832.6     98.7  98.4 97.5
2016   1,427.0 1,432.6 1,432.6     1,891.4 1,891.4 1,891.4     99.1  98.8 98.2
2017   1,474.3 1,474.3 1,474.3     1,888.2 1,888.2 1,888.2     99.8  99.3 99.1
2018   1,495.1 1,495.1 1,510.9     1,901.7 1,901.7 1,901.7    100.0 100.0 85.6
2019   1,473.0 1,473.0 1,473.0     1,759.2 1,759.2 1,759.2     98.5  97.7 85.5
2020   1,429.5 1,429.5 1,429.5     1,671.0 1,671.0 1,671.0     94.8  94.8 94.8
2021   1,357.4 1,357.4 1,357.4     1,357.4 1,357.4 1,357.4     93.1  90.9 86.8
2022   1,428.3 1,428.3 1,428.3     1,457.9 1,457.9 1,457.9     97.0  96.7 93.8
2023   1,080.0 1,080.0 1,080.0     1,080.0 1,080.0 1,080.0     96.7  96.3 82.5
2024   1,019.2 1,019.2 1,019.2     1,019.2 1,019.2 1,019.2     95.4  87.9 85.8
2025     880.1   880.1   880.1       880.1   880.1   880.1     99.3  99.3 99.0

In [7]:
spread = pd.DataFrame(
    {
        target: by_threshold[target].max(axis=1) - by_threshold[target].min(axis=1)
        for target in ("minimum", "daytime_low")
    }
)
print("Largest difference between thresholds in any year (MW):")
print(spread.max().round(1).to_string())
spread[spread.gt(0).any(axis=1)].round(1)

Largest difference between thresholds in any year (MW):
minimum       15.8
daytime_low    0.0


,minimum,daytime_low
year,,
2016,5.6,0.0
2018,15.8,0.0


## 4. The level steps between reporting years: a rescaled series

Ausgrid publishes one file per reporting year (1 May to 30 April), and the files do not always meet at the same level at midnight on 1 May (notebook 01). The steps are left uncorrected in the main series and stated as a limitation. This section asks how much they could matter, by building a second series in which the joins meet and repeating the whole backtest on it.

### 4.1 The rescaling

Each join's step is the change across midnight on 1 May less the usual midnight change of the nights around it. Every file is multiplied by one constant so that each measured step closes. The latest file keeps its level; each earlier file is brought to meet the file after it. The joins of 2021 and 2025 cannot be measured (no valid data in the hours around them) and get no adjustment.

**This is a bounding case, not a correction.** The steps chain, so the factors compound from the latest file back to the first. Closing every join treats each step as an error in the level of the earlier file, which is not known to be true: substations entering and leaving the kept set explain only part of the steps. The published series stays the main one.

The table gives, for each reporting-year file, the step measured at the join that ends it (per cent), the factor applied to the file, and the step that is left at the same join after rescaling (MW).

In [8]:
steps = clean.join_steps(total)
factors = clean.join_factors(steps)
rescaled = clean.rescale_reporting_years(total, factors)

rescaling = pd.DataFrame(
    {
        "step_at_end_pct": steps["step_pct"],
        "factor": factors,
        "step_left_mw": clean.join_steps(rescaled)["step_mw"],
    }
).rename_axis("reporting_year")
rescaling.to_csv(checks /"sensitivity_rescale_factors.csv")
print(rescaling.to_string(float_format="{:,.3f}".format))

                step_at_end_pct  factor  step_left_mw
reporting_year                                       
2015                      2.987   1.171         0.474
2016                      4.759   1.137         1.849
2017                      2.259   1.085         2.702
2018                      1.633   1.061         1.924
2019                     -0.480   1.044        -0.670
2020                     -1.065   1.049        -0.000
2021                        NaN   1.060           NaN
2022                      7.161   1.060         1.285
2023                     -4.121   0.990        -4.908
2024                      3.216   1.032         2.650
2025                        NaN   1.000           NaN
2026                        NaN   1.000           NaN


### 4.2 Annual lows under both series

Each year's whole-day minimum and daytime low (MW) on the published and on the rescaled series, and the rescaled value as a ratio of the published one. A calendar year draws on two reporting-year files, so the ratio is the factor of whichever file holds the year's lowest half hour.

In [9]:
lows = pd.concat({"published": annual_lows(total), "rescaled": annual_lows(rescaled)}, axis=1)
lows = lows.swaplevel(axis=1)[REPORTED]
for target in REPORTED:
    lows[target, "ratio"] = lows[target, "rescaled"] / lows[target, "published"]
lows = lows[REPORTED]
lows.to_csv(checks /"sensitivity_rescaled_lows.csv")
print(lows.to_string(float_format="{:,.3f}".format))

target daytime_low                   minimum                
         published  rescaled ratio published  rescaled ratio
year                                                        
2015     1,832.550 2,088.619 1.140 1,368.122 1,580.857 1.155
2016     1,891.371 2,113.973 1.118 1,432.613 1,621.073 1.132
2017     1,888.157 2,019.194 1.069 1,474.290 1,572.871 1.067
2018     1,901.739 1,985.734 1.044 1,495.080 1,577.606 1.055
2019     1,759.249 1,845.819 1.049 1,472.954 1,545.437 1.049
2020     1,670.980 1,754.551 1.050 1,429.476 1,499.819 1.049
2021     1,357.377 1,439.495 1.060 1,357.377 1,439.495 1.060
2022     1,457.912 1,442.796 0.990 1,428.305 1,442.796 1.010
2023     1,080.031 1,114.768 1.032 1,080.031 1,114.768 1.032
2024     1,019.176 1,019.176 1.000 1,019.176 1,019.176 1.000
2025       880.080   880.080 1.000   880.080   880.080 1.000


### 4.3 Table III under both series

All six models are refitted and replayed on the rescaled series, exactly as in notebook 05; PV capacity and weather are unchanged. This cell takes a few minutes.

One table per target, with the published series beside the rescaled one. The rescaled series is higher in the earlier years, so pinball losses in MW are not on quite the same scale between the two; the ranking of the models within a series, and the comparisons in section 4.4, are what to read.

In [10]:
forecasts_rescaled = evaluate.rolling_origin(rescaled, pv, weather, cfg)
forecasts_rescaled.to_parquet(interim / "forecasts_rescaled.parquet")


def under_both_series(target: str) -> pd.DataFrame:
    """Table III for one target on the published series beside the rescaled one."""
    on_rescaled = evaluate.score_table(forecasts_rescaled, target)
    on_rescaled.to_csv(checks /f"table3_{target}_rescaled.csv")
    on_published = evaluate.score_table(forecasts, target)
    both = pd.concat({"published": on_published[SHOWN], "rescaled": on_rescaled[SHOWN]}, axis=1)
    return both.swaplevel(axis=1)[SHOWN]


print("Daytime low: the target H1 and H2 are judged on")
under_both_series("daytime_low")

Daytime low: the target H1 and H2 are judged on


annual_pinball_mean          annual_coverage          monthly_pinball_mean          monthly_coverage         
                published rescaled       published rescaled            published rescaled        published rescaled
model                                                                                                              
M1                   63.1     42.4             0.4      0.4                 50.5     43.1              0.6      0.7
M2                   34.6     29.2             1.0      0.9                 53.5     38.4              0.7      0.7
M3                   43.3     87.0             0.7      0.0                 49.6     53.6              0.6      0.6
M4                   63.5    120.7             0.6      0.0                 51.8     64.2              0.6      0.5
M5                   28.5     20.8             0.7      0.9                 42.8     41.3              0.7      0.6
M6                   36.6     26.2             0.9      0.9                 75.1     73.2              0.5      0.5

In [11]:
print("Whole-day minimum: the headline target")
under_both_series("minimum")

Whole-day minimum: the headline target


annual_pinball_mean          annual_coverage          monthly_pinball_mean          monthly_coverage         
                published rescaled       published rescaled            published rescaled        published rescaled
model                                                                                                              
M1                   45.2     31.0             0.1      0.6                 43.5     35.1              0.4      0.6
M2                   31.1     22.9             1.0      1.0                 37.5     26.8              0.7      0.7
M3                   41.6     66.1             0.7      0.3                 41.4     40.2              0.6      0.5
M4                   60.0     98.9             0.6      0.1                 44.1     49.1              0.6      0.6
M5                   26.9     18.7             0.7      0.9                 32.5     25.5              0.6      0.8
M6                   28.0     19.6             0.9      0.9                 42.2     38.2              0.6      0.7

### 4.4 The comparisons under both series

The change in mean pinball loss (per cent) for each of the four comparisons, on the published and on the rescaled series. A comparison is robust to the level steps when it points the same way, with a similar size, on both.

In [12]:
def changes(scored: pd.DataFrame) -> pd.Series:
    """Change in mean pinball loss (%) for each comparison, reported target and level."""
    reported = scored[scored["target"].isin(REPORTED)]
    return evaluate.compare(reported, evaluate.COMPARISONS)["change_pct"]


by_series = pd.DataFrame(
    {"published": changes(forecasts), "rescaled": changes(forecasts_rescaled)}
)
by_series["same_direction"] = by_series["published"] * by_series["rescaled"] > 0
by_series.to_csv(checks /"sensitivity_rescaled_comparisons.csv")
by_series

published  rescaled  same_direction
comparison                          target      level                                       
H1: PV against time                 daytime_low annual        25.0     198.2            True
                                                monthly       -7.4      39.6           False
                                    minimum     annual        34.1     188.2            True
                                                monthly       10.3      49.8            True
H2: weather on top of PV            daytime_low annual        46.7      38.7            True
                                                monthly        4.5      19.8            True
                                    minimum     annual        44.2      49.6            True
                                                monthly        6.6      22.2            True
Model form: XGBoost against linear  daytime_low annual       -55.1     -82.8            True
                                                monthly      -17.3     -35.6            True
                                    minimum     annual       -55.2     -81.1            True
                                                monthly      -26.3     -48.2            True
Supplementary: weather in tree form daytime_low annual       -22.2     -20.8            True
                                                monthly      -43.0     -43.6            True
                                    minimum     annual        -4.0      -4.8            True
                                                monthly      -23.0     -33.4            True

## 5. Result

The three checks in one place: whether each comparison points the same way on the robustness target as on the whole-day minimum, how far any annual low moves with the reporting threshold, and whether each comparison points the same way on the rescaled series.

In [13]:
on_minimum = robust_changes.xs("minimum", axis=1, level="target")
on_robust = robust_changes.xs("robust", axis=1, level="target")
same_on_robust = on_minimum * on_robust > 0
print("1. Robustness target: comparisons pointing the same way as on the whole-day minimum")
print(same_on_robust.to_string())
print()
print("2. Reporting threshold: largest move of any annual low between 90%, 95% and 98% (MW)")
print(spread.max().round(1).to_string())
print()
first, last = factors.index[0], factors.index[-1]
print(f"3. Rescaled series: the FY{first} file is multiplied by {factors[first]:.3f} "
      f"and the FY{last} file by {factors[last]:.3f}")
held = by_series["same_direction"]
print(f"   Comparisons pointing the same way on both series: {held.sum()} of {len(held)}")
judged = by_series.xs("daytime_low", level="target")[["published", "rescaled"]]
print("   Change in mean pinball loss on the daytime low (%):")
print(judged.round(1).to_string())

1. Robustness target: comparisons pointing the same way as on the whole-day minimum
level                                annual  monthly
comparison                                          
H1: PV against time                    True     True
H2: weather on top of PV               True     True
Model form: XGBoost against linear     True     True
Supplementary: weather in tree form    True     True

2. Reporting threshold: largest move of any annual low between 90%, 95% and 98% (MW)
minimum       15.8
daytime_low    0.0

3. Rescaled series: the FY2015 file is multiplied by 1.171 and the FY2026 file by 1.000
   Comparisons pointing the same way on both series: 15 of 16
   Change in mean pinball loss on the daytime low (%):
                                             published  rescaled
comparison                          level                       
H1: PV against time                 annual        25.0     198.2
                                    monthly       -7.4      39.6
H2: weat